# Relation and Variable Graph From the Current Registries

This notebook inspects `RELATIONS` and `VARIABLES` directly, builds a graph with
`networkx`, and writes a rendered artifact without using removed plotting helpers.
        


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx

root = Path.cwd().resolve()
while root != root.parent and not (root / "src" / "fusdb").is_dir():
    root = root.parent

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from fusdb.registry import RELATIONS, VARIABLES


In [ ]:
relations = RELATIONS.get_filtered_relations()
relation_graph = nx.DiGraph()

for relation in relations:
    relation_node = f"relation::{relation.name}"
    relation_graph.add_node(relation_node, kind="relation", label=relation.name)

    for variable_name in relation.input_names:
        variable_node = f"variable::{variable_name}"
        relation_graph.add_node(variable_node, kind="variable", label=variable_name)
        relation_graph.add_edge(variable_node, relation_node)

    for variable_name in relation.output_names:
        variable_node = f"variable::{variable_name}"
        relation_graph.add_node(variable_node, kind="variable", label=variable_name)
        relation_graph.add_edge(relation_node, variable_node)

print(f"Registry variables: {len(list(VARIABLES._specs))}")
print(f"Graph nodes: {relation_graph.number_of_nodes()}")
print(f"Graph edges: {relation_graph.number_of_edges()}")
        


In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
positions = nx.spring_layout(relation_graph, seed=7, k=0.24)
relation_nodes = [node for node, data in relation_graph.nodes(data=True) if data["kind"] == "relation"]
variable_nodes = [node for node, data in relation_graph.nodes(data=True) if data["kind"] == "variable"]

nx.draw_networkx_nodes(relation_graph, positions, nodelist=relation_nodes, node_size=90, node_color="#d95f02", alpha=0.85, ax=ax)
nx.draw_networkx_nodes(relation_graph, positions, nodelist=variable_nodes, node_size=40, node_color="#1b9e77", alpha=0.70, ax=ax)
nx.draw_networkx_edges(relation_graph, positions, width=0.35, alpha=0.18, arrows=False, ax=ax)
ax.set_title("Relation-variable graph from current registries")
ax.set_axis_off()
plt.show()
        


In [ ]:
out_dir = root / "docs" / "code_docs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "relations_variables_graph.png"
fig.savefig(out_path, dpi=180, bbox_inches="tight")
print(f"Wrote: {out_path}")
        
